# 🎬 Video Model Hub — Colab Runner

این نوت‌بوک ماژول `app.video_gen` رو از ریپو **game-lead-finder** روی Colab اجرا میکنه.

**مراحل:**
1. سلول‌های نصب رو اجرا کن
2. مدل مورد نظر رو انتخاب کن
3. Prompt بده و تولید کن!

> ⚠️ **نیاز به Runtime نوع GPU** (T4 رایگان یا A100 در Pro). از منو: Runtime → Change runtime type → T4 GPU

## ۱. کلون ریپو و نصب پیش‌نیازها

In [ ]:
!git clone https://github.com/farzadabbasi617-star/game-lead-finder.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt
!pip install -q diffusers transformers accelerate imageio imageio-ffmpeg sentencepiece protobuf httpx
!pip install -q --upgrade torch torchvision

## ۲. چک کردن GPU

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## ۳. مرور مدل‌های موجود

In [ ]:
import sys
sys.path.insert(0, '/content/repo')
from app.video_gen import get_all_models, filter_models, registry_stats, VideoTaskType

print('📊 آمار کل رجیستری:')
import json
print(json.dumps(registry_stats(), indent=2, ensure_ascii=False))

print('\n🎯 مدل‌های مناسب Colab T2V (VRAM ≤ 16GB):')
for m in filter_models(task=VideoTaskType.TEXT_TO_VIDEO, max_vram_gb=16, free_only=True):
    print(f'  ▸ {m.id:25s} | {m.name:40s} | VRAM ~{m.vram_gb_min or "?"}GB | {m.max_seconds}s')

## ۴. تنظیم API Keys (اختیاری)

اگر میخوای از OpenRouter، HF Inference، Replicate یا fal استفاده کنی:

In [ ]:
import os
# os.environ['HF_TOKEN'] = 'hf_...'
# os.environ['OPENROUTER_API_KEY'] = 'sk-or-...'
# os.environ['REPLICATE_API_TOKEN'] = 'r8_...'
# os.environ['FAL_KEY'] = 'fal_...'

## ۵. تولید ویدیو — مثال ۱: CogVideoX-2B (سبک، مناسب T4)

In [ ]:
from app.video_gen import generate, GenerationRequest, VideoTaskType

req = GenerationRequest(
    model_id='cogvideox-2b',
    prompt='A cinematic shot of a golden dragon flying over a snowy mountain at sunset, epic scale, 4k',
    task=VideoTaskType.TEXT_TO_VIDEO,
    num_frames=49,
    num_inference_steps=50,
    guidance_scale=6.0,
    seed=42,
    output_dir='/content/outputs',
)
result = generate(req)
print('OK:', result.ok)
print('Elapsed:', result.elapsed_sec, 's')
print('Output:', result.output_path)
if result.error:
    print('Error:', result.error)

In [ ]:
# نمایش ویدیو
from IPython.display import Video
if result.ok:
    Video(result.output_path, embed=True, width=640)

## ۶. مثال ۲: LTX-Video (سریع‌ترین)

In [ ]:
req = GenerationRequest(
    model_id='ltx-video',
    prompt='A serene lake with pink cherry blossoms falling, gentle wind, morning light',
    num_frames=97,
    width=704, height=480,
    num_inference_steps=40,
    output_dir='/content/outputs',
)
result = generate(req)
print(result)
if result.ok:
    from IPython.display import Video
    Video(result.output_path, embed=True, width=640)

## ۷. مثال ۳: Image-to-Video با Stable Video Diffusion

In [ ]:
# آپلود عکس یا از URL دانلود کن
!wget -q https://huggingface.co/datasets/hf-internal-testing/diffusers-images/resolve/main/svd/rocket.png -O /content/input.png

req = GenerationRequest(
    model_id='stable-video-diffusion',
    prompt='',  # SVD needs no prompt
    task=VideoTaskType.IMAGE_TO_VIDEO,
    input_image_path='/content/input.png',
    output_dir='/content/outputs',
)
result = generate(req)
print(result)
if result.ok:
    from IPython.display import Video
    Video(result.output_path, embed=True, width=640)

## ۸. راه‌اندازی سرور FastAPI روی Colab (اختیاری)

اگر میخوای رابط وب و API کامل رو روی Colab بالا بیاری:

In [ ]:
!pip install -q pyngrok
from pyngrok import ngrok
# ngrok.set_auth_token('YOUR_NGROK_TOKEN')

import threading, uvicorn
from app.main import app

def run():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

threading.Thread(target=run, daemon=True).start()
public_url = ngrok.connect(8000)
print(f'🌐 UI:  {public_url}/video/')
print(f'📊 API: {public_url}/video/models')
print(f'📖 Docs: {public_url}/docs')

## ۹. Benchmark همه مدل‌های سبک روی T4

این سلول همه مدل‌های زیر 16GB VRAM رو با یک prompt یکسان تست میکنه:

In [ ]:
PROMPT = 'A cat playing piano in a jazz bar, cinematic lighting'
results = []
for m in filter_models(task=VideoTaskType.TEXT_TO_VIDEO, max_vram_gb=12, free_only=True):
    print(f'\n🎬 Trying {m.id}...')
    req = GenerationRequest(model_id=m.id, prompt=PROMPT,
                            num_frames=16, num_inference_steps=25,
                            output_dir='/content/bench')
    r = generate(req)
    results.append({'id': m.id, 'ok': r.ok, 'time_s': r.elapsed_sec,
                    'error': (r.error or '')[:80]})
    import gc, torch
    gc.collect(); torch.cuda.empty_cache()

import pandas as pd
pd.DataFrame(results)